# Gnomonic Expansion and Binomial Matrices
Peter Luschny, August 2026

### A first look

A 'gnomonic expansion' takes a stream of numbers and turns it into a growing rectangular table — a bit like building Pascal's Triangle row by row, except here each new number wraps an L-shaped border (a "gnomon", the ancient Greek term for the piece you add to a square to make it one size bigger) around the existing table, adding both a new row and a new column.

One side of that new border (a new row) is built by *adding up* the recent values (running sums), and the other side (a new column) is built by *subtracting* them (running differences). So a single flat sequence unfolds into a square matrix where sums and differences of the original numbers sit next to each other.

Illustrating the algorithm: Assume a 0-based sequence a = [1, 1, 2, 5, ...] and the first three steps already finished that led to the 3×3 matrix. Now, compute a(3) = 5 and place it at the lower end of the diagonal. Add a new row starting from there, adding the entry on the left in the row above and repeat this until reaching the first column: 5 + 2 -> 7 + 3 -> 10 + 5 -> 15. Next, add a new column, starting again at the new diagonal term, and subtract the term in the previous row to its left; repeat this up to the first row: 5 - 2 -> 3 - 1 -> 2 - 1 = 1.

            1   0   1  |  1      
            2   1   1  |  2      
            5   3   2  |  3      
            - - - - - - - -      
           15  10   7  |  5      

### The construction

Let $(a_n)_{n\ge 0}$ be a sequence of numbers. We define an increasing
sequence of square matrices $A^{(0)}\subset A^{(1)}\subset\cdots$, where
$A^{(n)}$ has size $n{+}1$, as follows.

Start with $A^{(0)} = (a_0)$. Given $A^{(n)}$, form $A^{(n+1)}$ by adding
one new row and one new column, both indexed $0,\dots,n$, sharing the value
$a_n$ at their common corner:


* New row, fill right to left: set $r_n=a_n$, and for
$k=n-1,\dots,0$,
$$
r_k = A^{(n)}_{n-1,\,k} + r_{k+1}.
$$
* New column, fill bottom to top: set $c_n = a_n$, and for
$i=n-1,\dots,0$,
$$
c_i = c_{i+1} - A^{(n)}_{i,\,n-1}.
$$

The new row becomes row $n$ of $A^{(n+1)}$, the new column becomes column
$n$, and all previously computed entries are left unchanged. We call this
process the *gnomonic expansion* of the sequence $a$.

### The formal definition

We associate a matrix $A(n, k), \, (n \ge 0, k \ge 0), $ with a 0-based sequence $ a(0), a(1), \ldots ,$ defined by 
$$ A(n, k) = \sum_{m=0}^d \binom{d}{m} \, s^{d - m}  a(p + m),$$
where $d = |n - k|$, $p = \min(n, k)$, and $s = 1$ if $k \le n$, otherwise $-1$. 

We call this matrix the *binomial matrix* of $a$. We also refer to the mapping $ a \mapsto A $ itself as the *gnomonic expansion* of $a$. The name describes the algorithmic, structural growth: each time a new term is appended to $a$, a new row is added below and a new column to the right of the existing matrix, transforming an $ n \times n $ matrix into an $ (n+1) \times (n+1) $ matrix. In this way, the given sequence $a$ is embedded as the main diagonal in the matrix. 

Further, for all $k\ge 0$,
$$
A(k,0) = \sum_{m=0}^{k}\binom{k}{m} a_m
\qquad\text{and}\qquad
A(0,k) = \sum_{m=0}^{k}\binom{k}{m}(-1)^{k-m} a_m .
$$

That is, the *first column is the binomial transform* of $ a$, and
the *first row the inverse binomial transform* of $ a$ (equivalently, the top row is the
sequence of iterated forward differences $\Delta^k a_0$).
This follows by induction: the recursive addition rule for the row is exactly Pascal's
rule $ \binom{d}{m}=\binom{d-1}{m-1}+\binom{d-1}{m} $ applied without sign,
while the subtraction rule for the column applies the same rule with
alternating sign.


Given its particular arithmetic and algorithmic simplicity, the gnomonic expansion of a sequence is one of the fundamental methods for investigating the structure of a sequence of numbers. Combined with the concept of iterators, as built into computer languages ​​such as Python, it can be implemented very efficiently, as we show below.

### OEIS

A399000 is the main reference with an *alternative* implementation, also for Maple and Mathematica.

* A398987 Lucas numbers
* A398988 Powers n^n
* A398989 Sets of lists 
* A398990 Partition numbers 
* A398991 Fubini numbers 
* A398992 Central binomial coefficients
* A398993 Bell numbers
* A398994 (big) Schröder numbers
* A398995 Pell numbers
* A398996 Motzkin numbers
* A398997 Number of involutions
* A398998 Euler numbers
* A398999 Factorial numbers
* A399000 Catalan numbers
* A399001 Fibonacci numbers
and
* A398133/A398134 Bernoulli numbers

## A Python class for gnomonic expansion

In [3]:
from collections.abc import Iterator
from fractions import Fraction as frac
type Seq = list[int | frac]
type Matrix = list[Seq]
type SeqIterator = Iterator[int | frac]

class GnomonicExpansion:
    def __init__(self, seq: SeqIterator, dim: int | None = None) -> None:
        self.seq_source = seq
        self.matrix: Matrix = []
        if dim is not None and dim > 0:
            self.grow(dim)
        else:
            # dim = None: In this case the matrix grows until the given iterator 
            # stops (seq.next() raises StopIteration). The constant 99 is merely
            # a safety measure and was chosen arbitrarily.
            self.grow(99) 

    def grow(self, steps: int = 1) -> None:

        for _ in range(steps):
            try:
                r = next(self.seq_source)
            except StopIteration:
                break

            n = len(self.matrix)

            if n == 0:
                self.matrix.append([r])
                continue

            # Build the new lower row from the previous lower row slice and r.
            lower_new = self.matrix[-1][:] + [r]
            for k in range(n, 0, -1):
                lower_new[k - 1] += lower_new[k]

            # Build the new upper column from the previous upper column slice and r.
            prev_upper = [self.matrix[i][n - 1] for i in range(n)]
            upper_new = [0] * n + [r]
            for i in range(n - 1, -1, -1):
                upper_new[i] = upper_new[i + 1] - prev_upper[i]

            # Append the new upper values to existing rows.
            for i in range(n):
                self.matrix[i].append(upper_new[i])

            # Append the new lower row.
            self.matrix.append(lower_new)

    def get_matrix(self) -> Matrix:
        """Returns the current state of the binomial matrix."""
        return self.matrix

    def print_matrix(self, fraction: bool = False) -> None:
     
        matrix = self.matrix
        n = len(matrix)

        for row in matrix:
            print('[', ", ".join(str(v) for v in row), ']')
        print()

        diags = [[matrix[d - i][i] for i in range(d + 1)] for d in range(n)]
        for dia in diags:
            print('[', ", ".join(str(v.numerator) for v in dia), ']')
        print()

        if fraction:
            for dia in diags:
                print('[', ", ".join(str(v.denominator) for v in dia), ']')

# Catalan numbers

In [4]:
def catalan_iterator() -> SeqIterator:
    c, n = 1, 0
    while True:
        yield c
        c = c * (4 * n + 2) // (n + 2)
        n += 1

In [5]:
cat = GnomonicExpansion(catalan_iterator(), 8)
cat.print_matrix()

[ 1, 0, 1, 1, 3, 6, 15, 36 ]
[ 2, 1, 1, 2, 4, 9, 21, 51 ]
[ 5, 3, 2, 3, 6, 13, 30, 72 ]
[ 15, 10, 7, 5, 9, 19, 43, 102 ]
[ 51, 36, 26, 19, 14, 28, 62, 145 ]
[ 188, 137, 101, 75, 56, 42, 90, 207 ]
[ 731, 543, 406, 305, 230, 174, 132, 297 ]
[ 2950, 2219, 1676, 1270, 965, 735, 561, 429 ]

[ 1 ]
[ 2, 0 ]
[ 5, 1, 1 ]
[ 15, 3, 1, 1 ]
[ 51, 10, 2, 2, 3 ]
[ 188, 36, 7, 3, 4, 6 ]
[ 731, 137, 26, 5, 6, 9, 15 ]
[ 2950, 543, 101, 19, 9, 13, 21, 36 ]



# Lucas numbers

In [4]:
def lucas_iterator() -> SeqIterator:
    a, b = 2, 1
    while True:
        yield a
        a, b = b, a + b

In [5]:
luc = GnomonicExpansion(lucas_iterator(), 8)
luc.print_matrix()

[ 2, -1, 3, -4, 7, -11, 18, -29 ]
[ 3, 1, 2, -1, 3, -4, 7, -11 ]
[ 7, 4, 3, 1, 2, -1, 3, -4 ]
[ 18, 11, 7, 4, 3, 1, 2, -1 ]
[ 47, 29, 18, 11, 7, 4, 3, 1 ]
[ 123, 76, 47, 29, 18, 11, 7, 4 ]
[ 322, 199, 123, 76, 47, 29, 18, 11 ]
[ 843, 521, 322, 199, 123, 76, 47, 29 ]

[ 2 ]
[ 3, -1 ]
[ 7, 1, 3 ]
[ 18, 4, 2, -4 ]
[ 47, 11, 3, -1, 7 ]
[ 123, 29, 7, 1, 3, -11 ]
[ 322, 76, 18, 4, 2, -4, 18 ]
[ 843, 199, 47, 11, 3, -1, 7, -29 ]



# Euler numbers

In [ ]:
from math import comb

def euler_iterator() -> SeqIterator:
    seq = [1]
    yield 1
    n = 1
    sign = -1
    while True:
        yield 0  # yield 0 for odd indices 
        val = sum(comb(2 * n, 2 * k) * seq[n - k] * (1 if k % 2 == 1 else -1)
              for k in range(1, n + 1))
        seq.append(val)
        yield sign * val
        sign = -sign
        n += 1

In [7]:
eul = GnomonicExpansion(euler_iterator(), 9)
eul.print_matrix()

[ 1, -1, 0, 2, 0, -16, 0, 272, 0 ]
[ 1, 0, -1, 2, 2, -16, -16, 272, 272 ]
[ 0, -1, -1, 1, 4, -14, -32, 256, 544 ]
[ -2, -2, -1, 0, 5, -10, -46, 224, 800 ]
[ 0, 2, 4, 5, 5, -5, -56, 178, 1024 ]
[ 16, 16, 14, 10, 5, 0, -61, 122, 1202 ]
[ 0, -16, -32, -46, -56, -61, -61, 61, 1324 ]
[ -272, -272, -256, -224, -178, -122, -61, 0, 1385 ]
[ 0, 272, 544, 800, 1024, 1202, 1324, 1385, 1385 ]

[ 1 ]
[ 1, -1 ]
[ 0, 0, 0 ]
[ -2, -1, -1, 2 ]
[ 0, -2, -1, 2, 0 ]
[ 16, 2, -1, 1, 2, -16 ]
[ 0, 16, 4, 0, 4, -16, 0 ]
[ -272, -16, 14, 5, 5, -14, -16, 272 ]
[ 0, -272, -32, 10, 5, -10, -32, 272, 0 ]



# Pell numbers

A000129, A016116, A007052, [A077957, A007070]

In mathematics, the Pell numbers are an infinite sequence of integers, known since ancient times, that comprise the denominators of the closest rational approximations to the square root of 2. This sequence of approximations begins ⁠
1/1⁠, ⁠3/2⁠, ⁠7/5⁠, ⁠17/12⁠, and ⁠41/29⁠, so the sequence of Pell numbers begins with 1, 2, 5, 12, and 29 (from Wikipedia).

In [19]:
def pell_iterator() -> SeqIterator:
    a, b = 0, 1
    while True:
        yield b
        a, b = b, a + 2 * b

In [20]:
pel = GnomonicExpansion(pell_iterator(), 9)
pel.print_matrix()

[ 1, 1, 2, 2, 4, 4, 8, 8, 16 ]
[ 3, 2, 3, 4, 6, 8, 12, 16, 24 ]
[ 10, 7, 5, 7, 10, 14, 20, 28, 40 ]
[ 34, 24, 17, 12, 17, 24, 34, 48, 68 ]
[ 116, 82, 58, 41, 29, 41, 58, 82, 116 ]
[ 396, 280, 198, 140, 99, 70, 99, 140, 198 ]
[ 1352, 956, 676, 478, 338, 239, 169, 239, 338 ]
[ 4616, 3264, 2308, 1632, 1154, 816, 577, 408, 577 ]
[ 15760, 11144, 7880, 5572, 3940, 2786, 1970, 1393, 985 ]

[ 1 ]
[ 3, 1 ]
[ 10, 2, 2 ]
[ 34, 7, 3, 2 ]
[ 116, 24, 5, 4, 4 ]
[ 396, 82, 17, 7, 6, 4 ]
[ 1352, 280, 58, 12, 10, 8, 8 ]
[ 4616, 956, 198, 41, 17, 14, 12, 8 ]
[ 15760, 3264, 676, 140, 29, 24, 20, 16, 16 ]



# Motzkin numbers

In [2]:
def motzkin_iterator() -> SeqIterator:
    a, b = 1, 1
    yield a
    yield b
    n = 2
    while True:
        m = ((2 * n + 1) * b + (3 * n - 3) * a) // (n + 2)
        yield m
        a, b, n = b, m, n + 1

In [3]:
mot = GnomonicExpansion(motzkin_iterator(), 9)
mot.print_matrix()

[ 1, 0, 1, 0, 2, 0, 5, 0, 14 ]
[ 2, 1, 1, 1, 2, 2, 5, 5, 14 ]
[ 5, 3, 2, 2, 3, 4, 7, 10, 19 ]
[ 14, 9, 6, 4, 5, 7, 11, 17, 29 ]
[ 42, 28, 19, 13, 9, 12, 18, 28, 46 ]
[ 132, 90, 62, 43, 30, 21, 30, 46, 74 ]
[ 429, 297, 207, 145, 102, 72, 51, 76, 120 ]
[ 1430, 1001, 704, 497, 352, 250, 178, 127, 196 ]
[ 4862, 3432, 2431, 1727, 1230, 878, 628, 450, 323 ]

[ 1 ]
[ 2, 0 ]
[ 5, 1, 1 ]
[ 14, 3, 1, 0 ]
[ 42, 9, 2, 1, 2 ]
[ 132, 28, 6, 2, 2, 0 ]
[ 429, 90, 19, 4, 3, 2, 5 ]
[ 1430, 297, 62, 13, 5, 4, 5, 0 ]
[ 4862, 1001, 207, 43, 9, 7, 7, 5, 14 ]



# Bell numbers

In [13]:
from itertools import accumulate

def bell_iterator() -> SeqIterator:
    row = [1]
    while True:
        yield row[0]
        row = list(accumulate([row[-1], *row]))

In [14]:
bell = GnomonicExpansion(bell_iterator(), 9)
bell.print_matrix()

[ 1, 0, 1, 1, 4, 11, 41, 162, 715 ]
[ 2, 1, 1, 2, 5, 15, 52, 203, 877 ]
[ 5, 3, 2, 3, 7, 20, 67, 255, 1080 ]
[ 15, 10, 7, 5, 10, 27, 87, 322, 1335 ]
[ 52, 37, 27, 20, 15, 37, 114, 409, 1657 ]
[ 203, 151, 114, 87, 67, 52, 151, 523, 2066 ]
[ 877, 674, 523, 409, 322, 255, 203, 674, 2589 ]
[ 4140, 3263, 2589, 2066, 1657, 1335, 1080, 877, 3263 ]
[ 21147, 17007, 13744, 11155, 9089, 7432, 6097, 5017, 4140 ]

[ 1 ]
[ 2, 0 ]
[ 5, 1, 1 ]
[ 15, 3, 1, 1 ]
[ 52, 10, 2, 2, 4 ]
[ 203, 37, 7, 3, 5, 11 ]
[ 877, 151, 27, 5, 7, 15, 41 ]
[ 4140, 674, 114, 20, 10, 20, 52, 162 ]
[ 21147, 3263, 523, 87, 15, 27, 67, 203, 715 ]



# Pólya Trees
A000081, A034781, A375467

In [8]:
from math import isqrt

def polyatree_iterator()  -> SeqIterator:
    yield 0
    yield 1
    t = [0, 1]; t_append = t.append
    D = [0, 1]; D_append = D.append
    i = 2

    while True:
        total = sum(t[i - j] * D[j] for j in range(1, i))
        t_i = total // (i - 1)
        t_append(t_i)
        yield t_i

        d_i = 0
        for d in range(1, isqrt(i) + 1):
            if i % d == 0:
                d_i += d * t[d]
                if d * d != i:
                    d2 = i // d
                    d_i += d2 * t[d2]
        D_append(d_i)
        i += 1

In [9]:
pti = GnomonicExpansion(polyatree_iterator(), 9)
pti.print_matrix()

[ 0, 1, -1, 2, -2, 4, -5, 13, -25 ]
[ 1, 1, 0, 1, 0, 2, -1, 8, -12 ]
[ 3, 2, 1, 1, 1, 2, 1, 7, -4 ]
[ 8, 5, 3, 2, 2, 3, 3, 8, 3 ]
[ 22, 14, 9, 6, 4, 5, 6, 11, 11 ]
[ 64, 42, 28, 19, 13, 9, 11, 17, 22 ]
[ 195, 131, 89, 61, 42, 29, 20, 28, 39 ]
[ 615, 420, 289, 200, 139, 97, 68, 48, 67 ]
[ 1991, 1376, 956, 667, 467, 328, 231, 163, 115 ]

[ 0 ]
[ 1, 1 ]
[ 3, 1, -1 ]
[ 8, 2, 0, 2 ]
[ 22, 5, 1, 1, -2 ]
[ 64, 14, 3, 1, 0, 4 ]
[ 195, 42, 9, 2, 1, 2, -5 ]
[ 615, 131, 28, 6, 2, 2, -1, 13 ]
[ 1991, 420, 89, 19, 4, 3, 1, 8, -25 ]



# Sets of Lists

In [10]:
def sets_of_lists_iterator() -> SeqIterator:
    b, a, n = 1, 1, 2
    yield b
    yield a

    while True:
        q = (2 * n - 1) * a - (n - 1) * (n - 2) * b
        b, a, n = a, q, n + 1
        yield q

In [11]:
sol = GnomonicExpansion(sets_of_lists_iterator(), 9)
sol.print_matrix()

[ 1, 0, 2, 6, 36, 240, 1920, 17640, 183120 ]
[ 2, 1, 2, 8, 42, 276, 2160, 19560, 200760 ]
[ 6, 4, 3, 10, 50, 318, 2436, 21720, 220320 ]
[ 26, 20, 16, 13, 60, 368, 2754, 24156, 242040 ]
[ 148, 122, 102, 86, 73, 428, 3122, 26910, 266196 ]
[ 1032, 884, 762, 660, 574, 501, 3550, 30032, 293106 ]
[ 8464, 7432, 6548, 5786, 5126, 4552, 4051, 33582, 323138 ]
[ 79592, 71128, 63696, 57148, 51362, 46236, 41684, 37633, 356720 ]
[ 842832, 763240, 692112, 628416, 571268, 519906, 473670, 431986, 394353 ]

[ 1 ]
[ 2, 0 ]
[ 6, 1, 2 ]
[ 26, 4, 2, 6 ]
[ 148, 20, 3, 8, 36 ]
[ 1032, 122, 16, 10, 42, 240 ]
[ 8464, 884, 102, 13, 50, 276, 1920 ]
[ 79592, 7432, 762, 86, 60, 318, 2160, 17640 ]
[ 842832, 71128, 6548, 660, 73, 368, 2436, 19560, 183120 ]



# Involutions (Young tableaux with n cells)

A000085, A005425, A123023, A378100

In [22]:
def involution_iterator() -> SeqIterator:
    a, b, n = 1, 1, 1
    yield a
    yield b

    while True:
        a, b = b, b + n * a
        n += 1
        yield b

In [23]:
inv = GnomonicExpansion(involution_iterator(), 9)
inv.print_matrix()

[ 1, 0, 1, 0, 3, 0, 15, 0, 105 ]
[ 2, 1, 1, 1, 3, 3, 15, 15, 105 ]
[ 5, 3, 2, 2, 4, 6, 18, 30, 120 ]
[ 14, 9, 6, 4, 6, 10, 24, 48, 150 ]
[ 43, 29, 20, 14, 10, 16, 34, 72, 198 ]
[ 142, 99, 70, 50, 36, 26, 50, 106, 270 ]
[ 499, 357, 258, 188, 138, 102, 76, 156, 376 ]
[ 1850, 1351, 994, 736, 548, 410, 308, 232, 532 ]
[ 7193, 5343, 3992, 2998, 2262, 1714, 1304, 996, 764 ]

[ 1 ]
[ 2, 0 ]
[ 5, 1, 1 ]
[ 14, 3, 1, 0 ]
[ 43, 9, 2, 1, 3 ]
[ 142, 29, 6, 2, 3, 0 ]
[ 499, 99, 20, 4, 4, 3, 15 ]
[ 1850, 357, 70, 14, 6, 6, 15, 0 ]
[ 7193, 1351, 258, 50, 10, 10, 18, 15, 105 ]



## LCM -- Least common multiple of \{1, 2, ..., n\} 

In [12]:
def LCM_iterator(lng: int) -> SeqIterator:
    if lng <= 0: 
        print("This iterator requires a positive run length.")
        return

    lambd = [1] * lng
    lcm = [1] * lng
    isp = [True] * lng

    for p in range(2, lng):
        if isp[p]:
            for i in range(p * p, lng, p):
                isp[i] = False
            k = p
            while k < lng:
                lambd[k] = p
                k *= p
        lcm[p] = lcm[p - 1] * lambd[p]

    yield from lcm

In [13]:
gen = LCM_iterator(8)
lcm = GnomonicExpansion(gen)
lcm.print_matrix()

[ 1, 0, 1, 2, -3, 44, -215, 1014 ]
[ 2, 1, 1, 3, -1, 41, -171, 799 ]
[ 5, 3, 2, 4, 2, 40, -130, 628 ]
[ 16, 11, 8, 6, 6, 42, -90, 498 ]
[ 53, 37, 26, 18, 12, 48, -48, 408 ]
[ 206, 153, 116, 90, 72, 60, 0, 360 ]
[ 757, 551, 398, 282, 192, 120, 60, 360 ]
[ 2780, 2023, 1472, 1074, 792, 600, 480, 420 ]

[ 1 ]
[ 2, 0 ]
[ 5, 1, 1 ]
[ 16, 3, 1, 2 ]
[ 53, 11, 2, 3, -3 ]
[ 206, 37, 8, 4, -1, 44 ]
[ 757, 153, 26, 6, 2, 41, -215 ]
[ 2780, 551, 116, 18, 6, 40, -171, 1014 ]



# Number of integer partitions

In [ ]:
def partitions_iterator() -> SeqIterator:
    p = [1]
    yield p[0]
    n = 1
    while True:
        total = 0
        k = 1
        while True:
            # generalized pentagonal number
            g1 = k * (3 * k - 1) // 2
            if g1 > n: break
            sign = 1 if k % 2 else -1
            total += sign * p[n - g1]
            g2 = k * (3 * k + 1) // 2
            if g2 <= n:
                total += sign * p[n - g2]
            k += 1
        p.append(total)
        yield total
        n += 1

In [12]:
pari = GnomonicExpansion(partitions_iterator(), 9)
pari.print_matrix()

[ 1, 0, 1, -1, 2, -4, 9, -21, 49 ]
[ 2, 1, 1, 0, 1, -2, 5, -12, 28 ]
[ 5, 3, 2, 1, 1, -1, 3, -7, 16 ]
[ 13, 8, 5, 3, 2, 0, 2, -4, 9 ]
[ 34, 21, 13, 8, 5, 2, 2, -2, 5 ]
[ 88, 54, 33, 20, 12, 7, 4, 0, 3 ]
[ 225, 137, 83, 50, 30, 18, 11, 4, 3 ]
[ 569, 344, 207, 124, 74, 44, 26, 15, 7 ]
[ 1425, 856, 512, 305, 181, 107, 63, 37, 22 ]

[ 1 ]
[ 2, 0 ]
[ 5, 1, 1 ]
[ 13, 3, 1, -1 ]
[ 34, 8, 2, 0, 2 ]
[ 88, 21, 5, 1, 1, -4 ]
[ 225, 54, 13, 3, 1, -2, 9 ]
[ 569, 137, 33, 8, 2, -1, 5, -21 ]
[ 1425, 344, 83, 20, 5, 0, 3, -12, 49 ]



# Bernoulli Matrix

In [10]:
def bernoulli_seidel() -> SeqIterator:
    """Generates Bernoulli numbers with B_1 = 1/2."""
    yield frac(1)     # B_0 = 1
    yield frac(1, 2)  # B_1 = 1/2

    ZERO = frac(0)  # odd-indexed Bernoulli number past B_1 vanish
    row = [1]       # B_0
    p2 = 8          # 2^(2+1) = 8
    m = 1           # row length

    while True:
        f = frac(row[-1], p2 - 2)
        yield -f if m % 2 == 0 else f  # yield B_n 
        yield ZERO  # yield B_{n+1} (always 0)
        row.append(0)
        p2 <<= 2
        m += 1
        for k in range(m - 2, -1, -1): row[k] += row[k + 1]
        for k in range(1, m): row[k] += row[k - 1]

In [11]:
ber = GnomonicExpansion(bernoulli_seidel(), 8)
ber.print_matrix(fraction=True)

[ 1, -1/2, 1/6, 0, -1/30, 0, 1/42, 0 ]
[ 3/2, 1/2, -1/3, 1/6, -1/30, -1/30, 1/42, 1/42 ]
[ 13/6, 2/3, 1/6, -1/6, 2/15, -1/15, -1/105, 1/21 ]
[ 3, 5/6, 1/6, 0, -1/30, 1/15, -8/105, 4/105 ]
[ 119/30, 29/30, 2/15, -1/30, -1/30, 1/30, -1/105, -4/105 ]
[ 5, 31/30, 1/15, -1/15, -1/30, 0, 1/42, -1/21 ]
[ 253/42, 43/42, -1/105, -8/105, -1/105, 1/42, 1/42, -1/42 ]
[ 7, 41/42, -1/21, -4/105, 4/105, 1/21, 1/42, 0 ]

[ 1 ]
[ 3, -1 ]
[ 13, 1, 1 ]
[ 3, 2, -1, 0 ]
[ 119, 5, 1, 1, -1 ]
[ 5, 29, 1, -1, -1, 0 ]
[ 253, 31, 2, 0, 2, -1, 1 ]
[ 7, 43, 1, -1, -1, -1, 1, 0 ]

[ 1 ]
[ 2, 2 ]
[ 6, 2, 6 ]
[ 1, 3, 3, 1 ]
[ 30, 6, 6, 6, 30 ]
[ 1, 30, 6, 6, 30, 1 ]
[ 42, 30, 15, 1, 15, 30, 42 ]
[ 1, 42, 15, 30, 30, 15, 42, 1 ]


# Use with an arbitrary sequence

You can wrap any list of integers with Python's 'iter' operator to use it with the Gnomonic Expansion class.

In [27]:
numbers = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
num_iteraor = iter(numbers)
seqit = GnomonicExpansion(num_iteraor)
seqit.print_matrix()

[ 2, 1, 1, -1, 3, -9, 23, -53, 115, -237 ]
[ 5, 3, 2, 0, 2, -6, 14, -30, 62, -122 ]
[ 13, 8, 5, 2, 2, -4, 8, -16, 32, -60 ]
[ 33, 20, 12, 7, 4, -2, 4, -8, 16, -28 ]
[ 83, 50, 30, 18, 11, 2, 2, -4, 8, -12 ]
[ 205, 122, 72, 42, 24, 13, 4, -2, 4, -4 ]
[ 495, 290, 168, 96, 54, 30, 17, 2, 2, 0 ]
[ 1169, 674, 384, 216, 120, 66, 36, 19, 4, 2 ]
[ 2707, 1538, 864, 480, 264, 144, 78, 42, 23, 6 ]
[ 6169, 3462, 1924, 1060, 580, 316, 172, 94, 52, 29 ]

[ 2 ]
[ 5, 1 ]
[ 13, 3, 1 ]
[ 33, 8, 2, -1 ]
[ 83, 20, 5, 0, 3 ]
[ 205, 50, 12, 2, 2, -9 ]
[ 495, 122, 30, 7, 2, -6, 23 ]
[ 1169, 290, 72, 18, 4, -4, 14, -53 ]
[ 2707, 674, 168, 42, 11, -2, 8, -30, 115 ]
[ 6169, 1538, 384, 96, 24, 2, 4, -16, 62, -237 ]

